# 03 — Multi-step Agent Pipeline

This notebook demonstrates a simple **perception → planning → action** pipeline.

**Objectives**
- Perception: turn raw state into features.
- Planning: pick an action using a short heuristic lookahead.
- Action: execute and observe the next state.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip -q install numpy matplotlib
else:
    print("Running locally. Install once with: pip install -r requirements.txt")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Environment setup

A 1D track with a goal near the end.

In [ ]:
class TrackEnv:
    def __init__(self, length=10, goal=8):
        self.length = length
        self.goal = goal
        self.reset()

    def reset(self):
        self.pos = 0
        return self.pos

    def step(self, action):  # action in {-1, +1}
        self.pos = int(np.clip(self.pos + action, 0, self.length - 1))
        done = self.pos == self.goal
        reward = 1.0 if done else -0.01
        return self.pos, reward, done, {}

## Pipeline components

In [ ]:
def perceive(obs, goal, length):
    return {
        "position": obs,
        "distance_to_goal": goal - obs,
        "progress": obs / (length - 1),
    }


def plan(features):
    # Simple heuristic lookahead: if goal is ahead, move right, else move left.
    return 1 if features["distance_to_goal"] > 0 else -1


def execute(env, action):
    return env.step(action)

## Run the pipeline

In [ ]:
env = TrackEnv(length=10, goal=8)
obs = env.reset()

trajectory = [obs]
actions = []

for t in range(20):
    features = perceive(obs, env.goal, env.length)
    action = plan(features)
    obs, reward, done, _ = execute(env, action)

    actions.append(action)
    trajectory.append(obs)

    print(
        f"t={t:02d} | obs={features['position']} | distance={features['distance_to_goal']} | "
        f"action={action:+d} | next_obs={obs}"
    )

    if done:
        print("Reached goal!")
        break

## Visualize decisions

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(trajectory, marker="o", label="Position")
plt.axhline(env.goal, linestyle="--", color="green", label="Goal")
plt.yticks(range(0, env.length))
plt.xlabel("Step")
plt.ylabel("Track position")
plt.title("Perception → Planning → Action trajectory")
plt.legend()
plt.tight_layout()
plt.show()

## Quick exercises

1. Modify `plan()` to avoid overshooting when close to the goal.
2. Add noise to observations in `perceive()` and evaluate robustness.
3. Replace the heuristic planner with a tiny search over two future steps.